# HarnessIR · SAM 3 Colab 五图验收与20图证据生成

这里只生成T4辅助证据，不生成图像编辑结果，也不算四工具完整验收。
使用你已下载的 **Meta原生SAM 3 `sam3.pt`**，不自动下载模型、不接受SAM3.1。
如果只有Hugging Face配置/tokenizer，或只有Transformers `model.safetensors`，请先取得原生官方 `sam3.pt`；不能直接改扩展名。

选择 **运行时 → 更改运行时类型 → GPU**。Python必须为3.12。部署会创建独立工具venv并安装锁定的Torch2.10/cu128；Colab内置Torch保持独立。

本Notebook配置 `colab_sam3_1008_v1`：使用官方1008输入、fp32、31类、阈值0.15/0.5、单图进程、2线程；与旧1024配置不同，标记formal_eligible=false，导回后建立新合同。
默认顺序：部署 → 5图 → 检查Mask/图例 → 明确填写reviewer → 20图 → 下载ZIP。

不需要Hugging Face令牌。推理进程禁网络/安装/下载。请将输入包与权重先上传Google Drive（推荐）或Colab `/content`；结果输出到Drive可避免运行时删除造成丢失。
官方源码固定：<https://github.com/facebookresearch/sam3/tree/0570b3a5be9c4e694f23d85232fb55f4a6f1f7fc>。


## 1. 配置文件路径
输入ZIP仅包含冻结20张原始LQ的canonical PNG，不含GT、增强图或凭据。请根据实际上传路径修改下面三个路径。

In [ ]:
from pathlib import Path
import sys
assert sys.version_info[:2] == (3,12), "请使用Python3.12运行时；不会自动替换环境"
# 若使用Google Drive，取消下一行的注释并完成登录：
# from google.colab import drive; drive.mount('/content/drive')
INPUT_ZIP = Path('/content/harnessir_sam3_inputs20.zip')
CHECKPOINT = Path('/content/sam3.pt')
OUTPUT = Path('/content/drive/MyDrive/HarnessIR/sam3_colab_1008_v1')
DEPLOYMENT = Path('/content/harnessir_sam3_deployment')
BUNDLE = Path('/content/harnessir_sam3_inputs20')
INPUT_ZIP_SHA256 = 'e0050ca6658826c23fc9e1ca994a357319d61b53ebd807a8fe1d51ad4c2631d8'
CHECKPOINT_SHA256 = ''  # 填本地原生sam3.pt的SHA256；不要仅使用Colab下载后自报的哈希
CHECKPOINT_SOURCE_URL = 'https://huggingface.co/facebook/sam3'
WALL_SECONDS = 7200  # 每次batch调用的上限；单图900秒，至少预留120秒保存
assert INPUT_ZIP.is_file(), '请先上传输入ZIP'
assert CHECKPOINT.is_file(), '请先上传原生sam3.pt权重'
assert len(CHECKPOINT_SHA256) == 64, '请从本地校验文件复制权重SHA256'
assert OUTPUT.parent.exists(), '输出到Drive时，请先挂载Drive；或改为/content下的目录'


## 2. 加载独立调度器、检查输入
此单元不导入Torch，不运行模型。

In [ ]:
RUNNER_SOURCE = '"""Standalone SAM3 Colab raw-handoff runner; no Controller/model API coupling.\n\nPreparation is explicit and network-enabled. Workers have network disabled,\none image/process, exact input/contract checks, and never install dependencies.\nRaw results require independent local import before becoming production cache.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport hashlib\nimport html\nimport io\nimport json\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport tarfile\nimport time\nimport traceback\nimport uuid\nimport zipfile\n\nREVISION = "0570b3a5be9c4e694f23d85232fb55f4a6f1f7fc"\nPROFILE = "colab_sam3_1008_v1"\nPARAMETERS = dict(model="SAM3", checkpoint_generation="SAM3-not-SAM3.1", device="cuda:0",\n                  precision="fp32", input_size=1008, source_view_long_edge=None,\n                  mask_threshold=0.5, score_threshold=0.15, timeout_seconds=900,\n                  cpu_threads=2, compile=False, tf32=False, roi_policy_id="full_frame_v1")\n\n\ndef sha(path):\n    with Path(path).open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                     ensure_ascii=False, allow_nan=False).encode()).hexdigest()\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef write(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temp = path.with_name(path.name + ".tmp")\n    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")\n    temp.replace(path)\n\n\ndef under(root, relative):\n    if "\\\\" in str(relative) or Path(relative).is_absolute():\n        raise ValueError("Only relative POSIX asset paths are allowed")\n    path = (Path(root) / relative).resolve()\n    if not path.is_relative_to(Path(root).resolve()):\n        raise ValueError("Asset path escapes its delivery directory")\n    return path\n\n\ndef extract_bundle(archive, destination, expected_sha):\n    if sha(archive) != expected_sha:\n        raise ValueError("Input package SHA256 mismatch")\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    with zipfile.ZipFile(archive) as source:\n        names = source.namelist()\n        if len(names) != len(set(n.casefold() for n in names)):\n            raise ValueError("Duplicate ZIP names")\n        if sum(i.file_size for i in source.infolist()) > 4 * 1024**3:\n            raise ValueError("Input ZIP exceeds 4 GiB budget")\n        for info in source.infolist():\n            path = under(destination, info.filename)\n            if info.is_dir() or ((info.external_attr >> 16) & 0o170000) == 0o120000:\n                raise ValueError("Directory/symlink members not expected")\n            path.parent.mkdir(parents=True, exist_ok=True)\n            with source.open(info) as incoming, path.open("wb") as outgoing:\n                import shutil\n                shutil.copyfileobj(incoming, outgoing)\n    return verify_inputs(destination)\n\n\ndef verify_inputs(bundle):\n    from PIL import Image\n    manifest = read(Path(bundle) / "inputs.json")\n    ids = set()\n    for row in manifest["images"]:\n        if row["image_id"] in ids or not re.fullmatch(r"[A-Za-z0-9_-]+", row["image_id"]):\n            raise ValueError("Invalid/duplicate image ID")\n        ids.add(row["image_id"])\n        path = under(bundle, row["canonical_path"])\n        if sha(path) != row["canonical_sha256"]:\n            raise ValueError("Canonical input hash changed")\n        with Image.open(path) as image:\n            if image.format != "PNG" or image.mode != "RGB" or image.size != (row["width"], row["height"]):\n                raise ValueError("Canonical image format/shape differs")\n    prompts = read(Path(bundle) / "prompt_set.json")\n    if len(prompts["labels"]) != 31 or len(set(prompts["labels"])) != 31 or prompts["unknown_id"] != 0:\n        raise ValueError("Expected exact 31-class prompt set")\n    for name in ("pilot5.json", "selection20.json"):\n        selected = read(Path(bundle) / name)["image_ids"]\n        if len(selected) != len(set(selected)) or not set(selected) <= ids:\n            raise ValueError("Selection differs from frozen inputs")\n    return manifest\n\n\ndef validate_result(output, row, contract_sha):\n    import numpy as np\n    output = Path(output)\n    metadata = read(output / "images" / (row["image_id"] + ".json"))\n    if metadata["status"] not in ("ok", "empty"):\n        raise ValueError("Failed/pending results cannot be reused")\n    for key in ("image_id", "canonical_sha256", "original_sha256", "canonicalizer_version", "width", "height"):\n        if metadata[key] != row[key]:\n            raise ValueError("Result/source binding differs: " + key)\n    if metadata["contract_sha256"] != contract_sha or metadata["coordinate_space"] != "canonical_pixels":\n        raise ValueError("Result configuration/coordinates differ")\n    if metadata.get("process_released") is not True:\n        raise ValueError("Worker release has not been verified")\n    path = under(output, metadata["npz_path"])\n    if sha(path) != metadata["npz_sha256"]:\n        raise ValueError("NPZ asset hash changed")\n    with np.load(path, allow_pickle=False) as arrays:\n        masks, boxes, scores, indices, labels = (arrays[k] for k in\n            ("masks", "boxes_xyxy", "scores", "prompt_indices", "labels"))\n        count = len(metadata["instances"])\n        if masks.shape != (count, row["height"], row["width"]) or masks.dtype != np.bool_:\n            raise ValueError("Instance mask shape/dtype mismatch")\n        if boxes.shape != (count, 4) or boxes.dtype != np.float32 or scores.shape != (count,) or scores.dtype != np.float32:\n            raise ValueError("Boxes/scores shape/dtype mismatch")\n        if indices.shape != (count,) or indices.dtype != np.int32 or labels.shape != (count,) or labels.dtype.kind != "U":\n            raise ValueError("Prompt association shape/dtype mismatch")\n        if not np.isfinite(boxes).all() or not np.isfinite(scores).all() or ((scores < 0) | (scores > 1)).any():\n            raise ValueError("Invalid finite scores/boxes")\n        if count and ((boxes[:, 0] < 0).any() or (boxes[:, 1] < 0).any() or\n                      (boxes[:, 2] > row["width"]).any() or (boxes[:, 3] > row["height"]).any() or\n                      (boxes[:, 2:] <= boxes[:, :2]).any()):\n            raise ValueError("Boxes outside canonical pixels or degenerate")\n        prompt_names = read(output / "prompt_set.json")["labels"]\n        seen = set()\n        for i, instance in enumerate(metadata["instances"]):\n            if not 0 <= int(indices[i]) < len(prompt_names):\n                raise ValueError("Invalid prompt index")\n            if str(labels[i]) != prompt_names[int(indices[i])] or instance["label"] != str(labels[i]) or instance["prompt_index"] != int(indices[i]):\n                raise ValueError("Mask/class/prompt mismatch")\n            if instance["id"] in seen or instance["array_index"] != i:\n                raise ValueError("Duplicate/reordered instance")\n            seen.add(instance["id"])\n            if not np.allclose(instance["bbox_xyxy"], boxes[i], rtol=0, atol=1e-4) or abs(instance["score"] - float(scores[i])) > 1e-6:\n                raise ValueError("JSON/NPZ instance values differ")\n        if (count == 0) != (metadata["status"] == "empty"):\n            raise ValueError("Empty detection status mismatch")\n    return metadata\n\n\ndef prepare(base):\n    """Explicit deployment only. Never invoked by inference or resume."""\n    import urllib.request\n    if sys.version_info[:2] != (3, 12):\n        raise RuntimeError("This pinned Notebook requires Python 3.12; no automatic runtime replacement")\n    base = Path(base).resolve()\n    base.mkdir(parents=True, exist_ok=True)\n    source = base / ("sam3-" + REVISION)\n    archive = base / "sam3_source.zip"\n    if not source.exists():\n        urllib.request.urlretrieve("https://codeload.github.com/facebookresearch/sam3/zip/" + REVISION, archive)\n        with zipfile.ZipFile(archive) as z:\n            for member in z.infolist():\n                path = under(base, member.filename)\n                if not path.is_relative_to(source):\n                    raise ValueError("Unexpected source ZIP root")\n                if ((member.external_attr >> 16) & 0o170000) == 0o120000:\n                    raise ValueError("Source ZIP symlink")\n                if member.is_dir():\n                    path.mkdir(parents=True, exist_ok=True)\n                else:\n                    path.parent.mkdir(parents=True, exist_ok=True)\n                    path.write_bytes(z.read(member))\n    python = base / "tool_env/bin/python"\n    if not python.exists():\n        subprocess.run([sys.executable, "-m", "venv", str(base / "tool_env")], check=True)\n    log = base / "deployment.log"\n    with log.open("a") as stream:\n        for arguments in (\n            ["setuptools==78.1.0", "wheel==0.45.1"],\n            ["--index-url", "https://download.pytorch.org/whl/cu128", "torch==2.10.0", "torchvision==0.25.0"],\n            ["numpy==1.26.4", "Pillow==12.1.0", "timm==1.0.22", "einops==0.8.1", "iopath==0.1.10",\n             "ftfy==6.1.1", "regex", "tqdm", "huggingface_hub", "psutil==7.2.2", "pycocotools==2.0.11", "scipy==1.15.3",\n             "opencv-python-headless==4.11.0.86", "hydra-core==1.3.2", "decord==0.6.0"],\n            ["--no-deps", "--no-build-isolation", str(source)],\n        ):\n            subprocess.run([str(python), "-m", "pip", "install", *arguments], check=True, stdout=stream, stderr=stream)\n        subprocess.run([str(python), "-m", "pip", "check"], check=True, stdout=stream, stderr=stream)\n    freeze = subprocess.check_output([str(python), "-m", "pip", "freeze", "--all"])\n    (base / "environment.freeze.txt").write_bytes(freeze)\n    witness_code = "import torch,json; torch.set_num_threads(2); torch.manual_seed(20261013); assert torch.cuda.is_available(); x=torch.arange(64,dtype=torch.float32,device=\'cuda\').reshape(8,8); y=x@x; torch.cuda.synchronize(); c=torch.arange(64,dtype=torch.float32).reshape(8,8); assert torch.equal(y.cpu(),c@c); print(json.dumps(dict(passed=True,torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0))))"\n    witness = json.loads(subprocess.check_output([str(python), "-c", witness_code], text=True).strip().splitlines()[-1])\n    components = [{"path": p.relative_to(source).as_posix(), "sha256": sha(p)} for p in sorted(source.rglob("*")) if p.is_file() and "__pycache__" not in p.parts and ".egg-info" not in str(p)]\n    receipt = dict(source_revision=REVISION, source_components=components, source_tree_sha256=digest(components),\n                   environment_lock_sha256=sha(base / "environment.freeze.txt"), python_executable=str(python),\n                   tokenizer_path=str(source / "sam3/assets/bpe_simple_vocab_16e6.txt.gz"),\n                   gpu_witness=witness, image_inference_completed=False, ready=False)\n    receipt["tokenizer_sha256"] = sha(receipt["tokenizer_path"])\n    receipt["python_version"] = sys.version\n    write(base / "deployment.json", receipt)\n    return receipt\n\n\ndef initialize(bundle, deployment, checkpoint, output, expected_checkpoint_sha, provenance_url):\n    import shutil\n    bundle, deployment, checkpoint, output = map(lambda p: Path(p).resolve(), (bundle, deployment, checkpoint, output))\n    verify_inputs(bundle)\n    if not checkpoint.is_file() or checkpoint.suffix != ".pt" or checkpoint.stat().st_size < 100_000_000:\n        raise ValueError("Official native SAM3 .pt weights required; configs/safetensors/SAM3.1 are not interchangeable")\n    if not re.fullmatch(r"[0-9a-f]{64}", expected_checkpoint_sha) or sha(checkpoint) != expected_checkpoint_sha:\n        raise ValueError("Checkpoint differs from independently recorded local hash")\n    receipt = read(deployment / "deployment.json")\n    if not receipt["gpu_witness"]["passed"] or receipt["source_revision"] != REVISION:\n        raise ValueError("Prepared environment/GPU witness differs")\n    if sha(deployment / "environment.freeze.txt") != receipt["environment_lock_sha256"] or sha(receipt["tokenizer_path"]) != receipt["tokenizer_sha256"]:\n        raise ValueError("Prepared assets changed")\n    source = deployment / ("sam3-" + REVISION)\n    for component in receipt["source_components"]:\n        if sha(under(source, component["path"])) != component["sha256"]:\n            raise ValueError("Prepared official source changed")\n    current_freeze = subprocess.check_output([receipt["python_executable"], "-m", "pip", "freeze", "--all"])\n    if hashlib.sha256(current_freeze).hexdigest() != receipt["environment_lock_sha256"]:\n        raise ValueError("Tool environment changed since preparation")\n    contract = dict(profile_id=PROFILE, parameters=PARAMETERS, checkpoint_sha256=expected_checkpoint_sha,\n                    source_revision=REVISION, source_tree_sha256=receipt["source_tree_sha256"],\n                    tokenizer_sha256=receipt["tokenizer_sha256"], environment_lock_sha256=receipt["environment_lock_sha256"],\n                    backend=receipt["gpu_witness"], prompt_set_sha256=digest(read(bundle / "prompt_set.json")))\n    if (output / "run.json").exists():\n        run = read(output / "run.json")\n        if run["contract"] != contract or run["inputs_sha256"] != sha(bundle / "inputs.json"):\n            raise ValueError("Existing output run has another contract/input list; use a new directory")\n        # Relocating identical assets after a Colab reset does not change contract.\n        run.update(checkpoint_path=str(checkpoint), bundle=str(bundle), deployment=str(deployment))\n        write(output / "run.json", run)\n        shutil.copyfile(deployment / "deployment.json", output / "deployment.json")\n        return run\n    output.mkdir(parents=True, exist_ok=True)\n    run = dict(handoff_version="sam3_raw_v1", run_id="sam3_" + uuid.uuid4().hex,\n               model="SAM3", checkpoint_generation="SAM3-not-SAM3.1", checkpoint_path=str(checkpoint),\n               checkpoint_sha256=expected_checkpoint_sha, checkpoint_provenance_url=provenance_url,\n               checkpoint_identity="user_supplied_native_pt_pending_strict_load", formal_eligible=False,\n               deviations=["Official SAM3 image input 1008 instead of prior profile 1024", "Colab backend/env locked independently"],\n               contract=contract, contract_sha256=digest(contract), inputs_sha256=sha(bundle / "inputs.json"),\n               bundle=str(bundle), deployment=str(deployment), image_inference_completed=False,\n               deployment_receipt_sha256=sha(deployment / "deployment.json"),\n               gpu_driver=subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], text=True).strip())\n    for name in ("inputs.json", "prompt_set.json", "pilot5.json", "selection20.json"):\n        shutil.copyfile(bundle / name, output / name)\n    shutil.copyfile(deployment / "environment.freeze.txt", output / "environment.freeze.txt")\n    shutil.copyfile(deployment / "deployment.json", output / "deployment.json")\n    shutil.copyfile(deployment / "deployment.log", output / "deployment.log")\n    write(output / "run.json", run)\n    return run\n\n\ndef offline():\n    os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", OMP_NUM_THREADS="2", MKL_NUM_THREADS="2", OPENBLAS_NUM_THREADS="2")\n    def audit(event, arguments):\n        if event in ("socket.connect", "socket.getaddrinfo", "socket.sendto", "socket.sendmsg"):\n            raise RuntimeError("Inference network is forbidden")\n        if event in ("subprocess.Popen", "os.system", "os.posix_spawn"):\n            import shlex\n            command = arguments[1] if event == "subprocess.Popen" else arguments[0]\n            tokens = [str(t).lower() for t in command] if isinstance(command, (list, tuple)) else shlex.split(str(command).lower())\n            if any(Path(t).stem in ("pip", "pip3", "uv", "conda", "curl", "wget", "git", "powershell", "pwsh") for t in tokens) or "-m pip" in " ".join(tokens):\n                raise RuntimeError("Inference package installation/download is forbidden")\n    sys.addaudithook(audit)\n\n\ndef worker(output, image_id):\n    offline()\n    import numpy as np\n    import torch\n    from PIL import Image\n    output = Path(output)\n    run = read(output / "run.json")\n    row = next(r for r in read(output / "inputs.json")["images"] if r["image_id"] == image_id)\n    source = under(run["bundle"], row["canonical_path"])\n    if sha(source) != row["canonical_sha256"] or sha(run["checkpoint_path"]) != run["checkpoint_sha256"]:\n        raise ValueError("Source/checkpoint changed before worker startup")\n    from sam3 import model_builder\n    from sam3.model.sam3_image_processor import Sam3Processor\n    # Official builder otherwise warns about missing weights and continues.\n    # Reject partial/incompatible checkpoint loading rather than emitting evidence.\n    def strict_load(model, checkpoint_path):\n        weights = torch.load(checkpoint_path, map_location="cpu", weights_only=True)\n        if isinstance(weights, dict) and isinstance(weights.get("model"), dict):\n            weights = weights["model"]\n        weights = {k.removeprefix("detector."): v for k, v in weights.items() if k.startswith("detector.")}\n        if not weights:\n            raise ValueError("No official SAM3 detector weights; incompatible format")\n        model.load_state_dict(weights, strict=True)\n    model_builder._load_checkpoint = strict_load\n    torch.set_num_threads(2)\n    torch.set_num_interop_threads(1)\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.cuda.reset_peak_memory_stats()\n    receipt = read(output / "deployment.json")\n    model = model_builder.build_sam3_image_model(checkpoint_path=run["checkpoint_path"],\n        bpe_path=receipt["tokenizer_path"], load_from_HF=False, device="cuda", eval_mode=True,\n        enable_inst_interactivity=False, compile=False)\n    # The official module sets TF32 at import, so set it again after imports.\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    processor = Sam3Processor(model, resolution=PARAMETERS["input_size"], device="cuda", confidence_threshold=PARAMETERS["score_threshold"])\n    labels = read(output / "prompt_set.json")["labels"]\n    all_masks, all_boxes, all_scores, all_indices, names, instances = [], [], [], [], [], []\n    with Image.open(source) as image, torch.inference_mode():\n        state = processor.set_image(image)\n        for index, label in enumerate(labels):\n            processor.reset_all_prompts(state)\n            result = processor.set_text_prompt(state=state, prompt=label)\n            masks, boxes, scores = result["masks_logits"], result["boxes"], result["scores"]\n            if not len(masks) == len(boxes) == len(scores):\n                raise ValueError("SAM3 mask/box/score count mismatch")\n            for i in range(len(scores)):\n                probabilities = masks[i].squeeze(0).detach().cpu().numpy()\n                if probabilities.shape != (row["height"], row["width"]) or not np.isfinite(probabilities).all() or probabilities.min() < 0 or probabilities.max() > 1:\n                    raise ValueError("Official masks_logits must contain full-resolution sigmoid probabilities")\n                native = boxes[i].detach().cpu().numpy().astype(np.float32)\n                box = np.clip(native, 0, [row["width"], row["height"], row["width"], row["height"]]).astype(np.float32)\n                score = float(scores[i].item())\n                masks_array = probabilities >= PARAMETERS["mask_threshold"]\n                all_masks.append(masks_array); all_boxes.append(box); all_scores.append(score)\n                all_indices.append(index); names.append(label)\n                instances.append(dict(id=f"prompt_{index:03d}_mask_{i:04d}", array_index=len(instances), label=label,\n                                      prompt_index=index, score=score, bbox_xyxy=box.tolist(), native_bbox_xyxy=native.tolist(),\n                                      area_pixels=int(masks_array.sum())))\n            del result, masks, boxes, scores\n    images = output / "images"\n    images.mkdir(exist_ok=True)\n    npz = images / (image_id + ".npz")\n    temporary = npz.with_suffix(".npz.tmp")\n    with temporary.open("wb") as stream:\n        np.savez_compressed(stream, masks=np.stack(all_masks) if all_masks else np.zeros((0, row["height"], row["width"]), dtype=bool),\n                            boxes_xyxy=np.asarray(all_boxes, dtype=np.float32).reshape(-1, 4), scores=np.asarray(all_scores, dtype=np.float32),\n                            prompt_indices=np.asarray(all_indices, dtype=np.int32), labels=np.asarray(names, dtype="U64"))\n    temporary.replace(npz)\n    torch.cuda.synchronize()\n    metadata = {k: row[k] for k in ("image_id", "canonical_sha256", "original_sha256", "canonicalizer_version", "width", "height")}\n    metadata.update(handoff_version="sam3_raw_v1", status="ok" if instances else "empty", coordinate_space="canonical_pixels",\n                    run_id=run["run_id"], contract_sha256=run["contract_sha256"], npz_path="images/" + npz.name,\n                    npz_sha256=sha(npz), instances=instances, checkpoint_strict_load=True,\n                    mask_values="binary_probability_ge_0.5", native_masks_may_overlap=True,\n                    peak_torch_allocated_mb=torch.cuda.max_memory_allocated()/1024**2,\n                    peak_torch_reserved_mb=torch.cuda.max_memory_reserved()/1024**2, process_released=False)\n    write(images / (image_id + ".json"), metadata)\n\n\ndef summary(output, planned):\n    output = Path(output)\n    rows = []\n    for image_id in planned:\n        path = output / "images" / (image_id + ".json")\n        value = read(path) if path.exists() else dict(image_id=image_id, status="pending")\n        rows.append({k: value.get(k) for k in ("image_id", "status", "error_code", "elapsed_seconds", "npz_path", "log_path", "peak_rss_mb", "peak_gpu_memory_mb", "process_released")})\n    with (output / "processing_status.csv.tmp").open("w", newline="", encoding="utf-8") as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]) if rows else ["image_id", "status"])\n        writer.writeheader(); writer.writerows(rows)\n    (output / "processing_status.csv.tmp").replace(output / "processing_status.csv")\n    result = {status: sum(r["status"] == status for r in rows) for status in ("ok", "empty", "failed", "pending")}\n    result.update(planned=len(planned), complete=result["pending"] == result["failed"] == 0, formal_eligible=False)\n    write(output / "summary.json", result)\n    return result\n\n\ndef pilot_digest(output):\n    output = Path(output)\n    run = read(output / "run.json")\n    rows = {r["image_id"]: r for r in read(output / "inputs.json")["images"]}\n    ids = read(output / "pilot5.json")["image_ids"]\n    values = []\n    for image_id in ids:\n        metadata = validate_result(output, rows[image_id], run["contract_sha256"])\n        values.append(dict(image_id=image_id, metadata_sha256=sha(output / "images" / (image_id + ".json")), npz_sha256=metadata["npz_sha256"]))\n    return digest(dict(run_id=run["run_id"], contract_sha256=run["contract_sha256"], results=values))\n\n\ndef accept_pilot(output, reviewer):\n    if not str(reviewer).strip():\n        raise ValueError("Reviewer must be named after inspecting masks/legend")\n    write(Path(output) / "pilot_review.json", dict(accepted=True, reviewed_by=reviewer,\n          pilot_digest=pilot_digest(output), review_scope="T4_only_not_all_four_tools_gate"))\n\n\ndef batch(output, stage, wall_seconds=7200, retry_failed=False):\n    # Linux advisory lock is held for the entire dispatch, released on interrupt.\n    import fcntl\n    output = Path(output)\n    with (output / "dispatch.lock").open("a") as stream:\n        fcntl.flock(stream, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        return _batch(output, stage, wall_seconds, retry_failed)\n\n\ndef _batch(output, stage, wall_seconds=7200, retry_failed=False):\n    import psutil\n    output = Path(output).resolve()\n    run = read(output / "run.json")\n    if stage not in ("pilot", "selection20"):\n        raise ValueError("Unknown batch stage")\n    verify_inputs(run["bundle"])\n    rows = {r["image_id"]: r for r in read(output / "inputs.json")["images"]}\n    lease_path = output / "worker_lease.json"\n    if lease_path.exists():\n        lease = read(lease_path)\n        if psutil.pid_exists(lease["pid"]):\n            try:\n                command = psutil.Process(lease["pid"]).cmdline()\n                if "worker" in command and str(output) in command:\n                    raise RuntimeError("Previous worker still running after interruption; wait or terminate it explicitly")\n            except (psutil.NoSuchProcess, psutil.AccessDenied):\n                pass\n    ids = read(output / ("pilot5.json" if stage == "pilot" else "selection20.json"))["image_ids"]\n    if stage != "pilot":\n        review = read(output / "pilot_review.json")\n        if review.get("accepted") is not True or review["pilot_digest"] != pilot_digest(output):\n            raise ValueError("Current real five-image T4 pilot must be reviewed first")\n    python = read(output / "deployment.json")["python_executable"]\n    actual_freeze = subprocess.check_output([python, "-m", "pip", "freeze", "--all"])\n    if hashlib.sha256(actual_freeze).hexdigest() != run["contract"]["environment_lock_sha256"]:\n        raise ValueError("Tool environment differs; do not reuse/start jobs")\n    deadline = time.monotonic() + wall_seconds\n    planned = read(output / "selection20.json")["image_ids"]\n    (output / "logs").mkdir(exist_ok=True)\n    (output / "images").mkdir(exist_ok=True)\n    for image_id in ids:\n        path = output / "images" / (image_id + ".json")\n        if path.exists():\n            value = read(path)\n            if value["status"] in ("ok", "empty"):\n                if value.get("process_released") is True:\n                    validate_result(output, rows[image_id], run["contract_sha256"])\n                    print(image_id, "validated cache hit", flush=True)\n                    continue\n                value.update(status="failed", error_code="interrupted_unverified_release", error_message="Worker result was never finalized by Controller; explicit retry required")\n                write(path, value)\n            if not retry_failed:\n                print(image_id, "previous failure retained; explicit retry required", flush=True)\n                continue\n            attempts = value.get("attempt", 1)\n            if attempts >= 2:\n                print(image_id, "limited retry exhausted", flush=True)\n                continue\n            backup = path.with_name(path.stem + f".attempt{attempts}.json")\n            if not backup.exists():\n                path.replace(backup)\n        else:\n            attempts = 0\n        if deadline - time.monotonic() < PARAMETERS["timeout_seconds"] + 120:\n            print("Wall budget insufficient for another job and save reserve", flush=True)\n            break\n        active = subprocess.check_output(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader,nounits"], text=True).strip()\n        if active:\n            raise RuntimeError("GPU already has an active compute process; single-worker policy refuses concurrent startup")\n        started = time.monotonic(); peak = 0.; samples = []\n        gpu_peak, gpu_last, gpu_value = None, -1., None\n        logfile = output / "logs" / f"{image_id}.attempt{attempts+1}.log"\n        error = None\n        interruption = None\n        with logfile.open("w") as log:\n            process = subprocess.Popen([python, str(Path(__file__).resolve()), "worker", "--output", str(output), "--image-id", image_id],\n                                       stdout=log, stderr=subprocess.STDOUT, start_new_session=True)\n            write(lease_path, dict(pid=process.pid, image_id=image_id, run_id=run["run_id"]))\n            try:\n                while process.poll() is None:\n                    try:\n                        parent = psutil.Process(process.pid)\n                        rss = sum(p.memory_info().rss for p in [parent, *parent.children(recursive=True)] if p.is_running()) / 1024**2\n                        peak = max(peak, rss)\n                        if time.monotonic()-gpu_last >= 1:\n                            try:\n                                queried = subprocess.check_output(["nvidia-smi", "--query-compute-apps=pid,used_gpu_memory", "--format=csv,noheader,nounits"], text=True, timeout=3)\n                                pidset = {p.pid for p in [parent, *parent.children(recursive=True)]}\n                                values = [float(line.split(",")[1].strip()) for line in queried.splitlines()\n                                          if int(line.split(",")[0].strip()) in pidset and line.split(",")[1].strip().replace(".", "", 1).isdigit()]\n                                gpu_value = sum(values) if values else None\n                                if gpu_value is not None: gpu_peak = max(gpu_peak or 0., gpu_value)\n                            except (subprocess.SubprocessError, ValueError, OSError):\n                                gpu_value = None\n                            gpu_last = time.monotonic()\n                        samples.append(dict(seconds=time.monotonic()-started, rss_mb=rss, gpu_process_memory_mb=gpu_value))\n                        if rss > 24576 or psutil.virtual_memory().available < 1024**3:\n                            error = "memory_budget"\n                    except psutil.Error:\n                        pass\n                    if time.monotonic()-started > PARAMETERS["timeout_seconds"]:\n                        error = "timeout"\n                    if error:\n                        break\n                    time.sleep(0.2)\n            except BaseException as exc:\n                interruption = exc\n                error = "interrupted"\n            finally:\n                if process.poll() is None:\n                    import signal\n                    try: os.killpg(process.pid, signal.SIGTERM)\n                    except ProcessLookupError: pass\n                    try: process.wait(timeout=10)\n                    except subprocess.TimeoutExpired:\n                        os.killpg(process.pid, signal.SIGKILL); process.wait()\n            process.wait()\n        write(output / "logs" / f"{image_id}.attempt{attempts+1}.resources.json", samples)\n        elapsed = time.monotonic()-started\n        released = not psutil.pid_exists(process.pid)\n        if process.returncode != 0 or error:\n            text = logfile.read_text(errors="replace")\n            code = error or ("oom" if "out of memory" in text.lower() else\n                             "import_error" if "ModuleNotFoundError" in text or "ImportError" in text else\n                             "model_missing" if "FileNotFoundError" in text else\n                             "checkpoint_incompatible" if "load_state_dict" in text or "incompatible format" in text else "runtime_error")\n            metadata = {k: rows[image_id][k] for k in ("image_id", "canonical_sha256", "original_sha256", "canonicalizer_version", "width", "height")}\n            metadata.update(status="failed", error_code=code, error_message=text[-3000:], run_id=run["run_id"],\n                            contract_sha256=run["contract_sha256"], instances=[], attempt=attempts+1)\n        else:\n            metadata = read(path)\n            metadata["attempt"] = attempts+1\n        metadata.update(elapsed_seconds=elapsed, peak_rss_mb=peak if samples else None, peak_gpu_memory_mb=gpu_peak,\n                        gpu_memory_measurement="nvidia-smi process sample every 1s; torch allocator peak additionally in success metadata",\n                        process_released=released, log_path=logfile.relative_to(output).as_posix(), returncode=process.returncode)\n        write(path, metadata)\n        if metadata["status"] in ("ok", "empty"):\n            try:\n                validate_result(output, rows[image_id], run["contract_sha256"])\n            except Exception as exc:\n                metadata.update(status="failed", error_code="output_validation", error_message=str(exc))\n                write(path, metadata)\n        summary(output, planned)\n        print(image_id, metadata["status"], round(elapsed, 2), "seconds", flush=True)\n        if interruption:\n            raise interruption\n    total = summary(output, planned)\n    states = [read(output / "images" / (i + ".json"))["status"] if (output / "images" / (i + ".json")).exists() else "pending" for i in ids]\n    result = {status: states.count(status) for status in ("ok", "empty", "failed", "pending")}\n    result.update(stage=stage, planned=len(ids), complete=result["failed"] == result["pending"] == 0, selection20=total, formal_eligible=False)\n    write(output / ("pilot_status.json" if stage == "pilot" else "selection20_status.json"), result)\n    return result\n\n\ndef gallery(output):\n    import base64\n    import numpy as np\n    from PIL import Image, ImageDraw\n    output = Path(output)\n    run = read(output / "run.json")\n    labels = read(output / "prompt_set.json")["labels"]\n    colors = [tuple(64 + b % 192 for b in hashlib.sha256(f"0:{name}".encode()).digest()[:3]) for name in labels]\n    rows = {r["image_id"]: r for r in read(output / "inputs.json")["images"]}\n    def encoded(image):\n        image.thumbnail((900, 900))\n        stream = io.BytesIO(); image.save(stream, format="PNG")\n        return "data:image/png;base64," + base64.b64encode(stream.getvalue()).decode()\n    cards = []\n    for image_id in read(output / "selection20.json")["image_ids"]:\n        row = rows[image_id]\n        image = Image.open(under(run["bundle"], row["canonical_path"])).convert("RGB")\n        path = output / "images" / (image_id + ".json")\n        status = read(path)["status"] if path.exists() else "pending"\n        overlay = image.copy(); found = []\n        if status in ("ok", "empty"):\n            metadata = validate_result(output, row, run["contract_sha256"])\n            with np.load(under(output, metadata["npz_path"]), allow_pickle=False) as z:\n                pixels = np.asarray(overlay).astype(np.float32)\n                for mask, index, box, score in zip(z["masks"], z["prompt_indices"], z["boxes_xyxy"], z["scores"]):\n                    pixels[mask] = pixels[mask]*0.55 + np.asarray(colors[int(index)])*0.45\n                    found.append(f"{labels[int(index)]}: {float(score):.3f}")\n                overlay = Image.fromarray(pixels.clip(0,255).astype(np.uint8))\n                draw = ImageDraw.Draw(overlay)\n                for box, index in zip(z["boxes_xyxy"], z["prompt_indices"]):\n                    draw.rectangle(box.tolist(), outline=colors[int(index)], width=3)\n        cards.append(f"<section><h2>{html.escape(image_id)} — {html.escape(status)}</h2><img src=\'{encoded(image)}\'><img src=\'{encoded(overlay)}\'><p>{html.escape(\', \'.join(found))}</p></section>")\n    legend = " ".join(f"<span style=\'background:rgb{color};padding:4px;color:black\'>{html.escape(label)}</span>" for color,label in zip(colors,labels))\n    page = "<!doctype html><meta charset=\'utf-8\'><title>HarnessIR SAM3 evidence</title><style>body{font:16px sans-serif;background:#eee;padding:20px}section{background:white;padding:15px;margin:20px 0}img{max-width:48%;vertical-align:top}span{display:inline-block;margin:3px}</style><h1>SAM3证据：原图 / Mask叠加</h1><p>工具证据，非恢复结果；重叠实例仍保留在NPZ。Formal eligible: false.</p>" + legend + "".join(cards)\n    (output / "evidence_gallery.html").write_text(page, encoding="utf-8")\n    return output / "evidence_gallery.html"\n\n\ndef export_results(output, destination):\n    output = Path(output).resolve(); destination = Path(destination).resolve()\n    if destination.is_relative_to(output):\n        raise ValueError("Result ZIP must be outside output directory")\n    run = read(output / "run.json")\n    for row in read(output / "inputs.json")["images"]:\n        path = output / "images" / (row["image_id"] + ".json")\n        if path.exists() and read(path)["status"] in ("ok", "empty"):\n            validate_result(output, row, run["contract_sha256"])\n    gallery(output)\n    files = [p for p in sorted(output.rglob("*")) if p.is_file() and not p.name.endswith(".tmp")]\n    index = [{"path":p.relative_to(output).as_posix(), "sha256":sha(p), "bytes":p.stat().st_size} for p in files]\n    write(output / "delivery_manifest.json", dict(kind="sam3_raw_handoff_not_production_cache", files=index, formal_eligible=False))\n    with zipfile.ZipFile(destination.with_suffix(".zip.tmp"), "w", zipfile.ZIP_DEFLATED) as z:\n        for p in [*files, output / "delivery_manifest.json"]:\n            z.write(p, p.relative_to(output).as_posix())\n    destination.with_suffix(".zip.tmp").replace(destination)\n    return dict(path=str(destination), sha256=sha(destination), bytes=destination.stat().st_size)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("action", choices=("prepare", "worker", "batch"))\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--image-id")\n    parser.add_argument("--stage", choices=("pilot", "selection20"), default="pilot")\n    args = parser.parse_args()\n    try:\n        if args.action == "prepare": print(json.dumps(prepare(args.output)))\n        elif args.action == "worker": worker(args.output, args.image_id)\n        else: print(json.dumps(batch(args.output, args.stage)))\n    except Exception:\n        traceback.print_exc(); return 1\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
RUNNER_PATH = Path('/content/sam3_handoff.py')
RUNNER_PATH.write_text(RUNNER_SOURCE, encoding='utf-8')
import importlib.util
spec = importlib.util.spec_from_file_location('sam3_handoff', RUNNER_PATH)
hir = importlib.util.module_from_spec(spec); spec.loader.exec_module(hir)
manifest = hir.extract_bundle(INPUT_ZIP, BUNDLE, INPUT_ZIP_SHA256)
print('已核对输入', len(manifest['images']), '张；无模型推理')


## 3. 显式部署环境
这是唯一允许安装依赖和下载官方源码的步骤。日志保存到 deployment.log。失败时查看日志，不能改用其他模型或精度。部署后执行真实CUDA计算检查；这不等于图片推理验收。

In [ ]:
receipt = hir.prepare(DEPLOYMENT)
print(receipt['gpu_witness'])
run = hir.initialize(BUNDLE, DEPLOYMENT, CHECKPOINT, OUTPUT, CHECKPOINT_SHA256, CHECKPOINT_SOURCE_URL)
print('配置合同：', run['contract_sha256'])


## 4. 先跑五张真实图
每张启动独立模型进程，用完退出。失败保留日志；无检测则生成合法空数组。中断会终止子进程并记录。

In [ ]:
pilot = hir.batch(OUTPUT, 'pilot', wall_seconds=WALL_SECONDS)
print(pilot)


## 5. 检查证据与图例
打开完整HTML查看原图/Mask叠加：重点检查人、文字、细小对象、边缘对齐和跨类别重叠。空检测不能证明不存在对象。显示pending的图尚未运行。

In [ ]:
from IPython.display import display, HTML
gallery_path = hir.gallery(OUTPUT)
display(HTML(gallery_path.read_text(encoding='utf-8')))
print('已保存：', gallery_path)


## 6. 五图全部通过后，手动确认
只有真实五图输出可读、类别/坐标正确且你已看图，才填写自己的名称。这个review只覆盖T4，不能代替本地四工具验收。若不通过，先导出失败记录，不要继续20图。

In [ ]:
REVIEWED_BY = ''  # 看过五图后填写你的名称；不要自动填充
assert REVIEWED_BY.strip(), '先检查五图Mask，再填写审核人'
hir.accept_pilot(OUTPUT, REVIEWED_BY)
print('T4五图已绑定审核，允许继续20图')


## 7. 继续20张
复用严格一致的五图结果，只处理其他15张。已有失败默认保留；若原因已经修复，明确设置retry_failed=True，最多两次实际尝试。参数/权重/环境变化必须用新输出目录。

In [ ]:
result20 = hir.batch(OUTPUT, 'selection20', wall_seconds=WALL_SECONDS, retry_failed=False)
print(result20)


## 8. 导出并下载
未全部完成也可导出真实阶段结果；summary明确列出失败和pending，不宣称完成。ZIP含NPZ/JSON、实际环境锁、日志、资源记录与自包含HTML；不包含权重/令牌。

In [ ]:
RESULT_ZIP = Path('/content/harnessir_sam3_results.zip')
receipt = hir.export_results(OUTPUT, RESULT_ZIP)
print(receipt)
from google.colab import files
files.download(str(RESULT_ZIP))


下载后将ZIP交回本地，待校验后解包到 `Data/annotations/incoming/t4_sam3/<run_id>/`。该ZIP是原始handoff，尚不是正式cache；我负责合同绑定、严格验证和正式缓存转换。

恢复操作：重新挂载同一Drive、上传相同输入包与权重，运行1–3再运行4或7。完全一致结果才命中；受损结果会报错而不是被当作有效缓存。